# Stochastic Portfolio Optimization using Monte Carlo Simulations

Exploratory notebook version of the pipeline in `src/`. This walks through data loading, the Monte Carlo simulation, the efficient frontier, VaR/CVaR risk metrics, and the backtest vs the benchmark — with plots rendered inline.

See the top-level `README.md` for full project context and results.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_price_data, TICKERS, BENCHMARK_NAME
from monte_carlo import daily_log_returns, run_monte_carlo, best_portfolios
from risk_metrics import backtest_summary

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Load price data

Uses the synthetic offline dataset by default (see `data_loader.py` to switch to live `yfinance` data).

In [ ]:
prices, is_synthetic = load_price_data()
asset_cols = [c for c in prices.columns if c != BENCHMARK_NAME]
print('Synthetic data:', is_synthetic)
print('Universe:', asset_cols)
prices.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
(prices / prices.iloc[0] * 100).plot(ax=ax)
ax.set_title('Indexed Price History (Base = 100)')
ax.set_ylabel('Indexed Price')
plt.show()

## 2. Daily log returns

In [ ]:
asset_returns = daily_log_returns(prices[asset_cols])
bench_returns = daily_log_returns(prices[[BENCHMARK_NAME]])
asset_returns.describe()

## 3. Monte Carlo simulation

Randomly samples long-only, fully-invested portfolio weights and evaluates each on annualized return, volatility, and Sharpe ratio. Fully vectorized — scales to millions of portfolios.

In [ ]:
mc = run_monte_carlo(asset_returns, n_simulations=500_000, seed=7)
best = best_portfolios(mc)
max_sharpe, min_vol = best['max_sharpe'], best['min_volatility']

print(f"Max Sharpe portfolio -> return={max_sharpe['exp_return']*100:.2f}% "
      f"vol={max_sharpe['volatility']*100:.2f}% sharpe={max_sharpe['sharpe']:.3f}")
print(f"Min Vol portfolio    -> return={min_vol['exp_return']*100:.2f}% "
      f"vol={min_vol['volatility']*100:.2f}% sharpe={min_vol['sharpe']:.3f}")

## 4. Efficient frontier

In [ ]:
idx = np.random.default_rng(1).choice(len(mc['sharpe_ratios']), size=20_000, replace=False)
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(mc['volatilities'][idx]*100, mc['exp_returns'][idx]*100,
                 c=mc['sharpe_ratios'][idx], cmap='viridis', s=6, alpha=0.5)
plt.colorbar(sc, label='Sharpe Ratio')
ax.scatter(max_sharpe['volatility']*100, max_sharpe['exp_return']*100,
           marker='*', s=400, color='crimson', edgecolor='black', label='Max Sharpe')
ax.scatter(min_vol['volatility']*100, min_vol['exp_return']*100,
           marker='D', s=120, color='gold', edgecolor='black', label='Min Vol')
ax.set_xlabel('Annualized Volatility (%)'); ax.set_ylabel('Annualized Return (%)')
ax.set_title('Efficient Frontier'); ax.legend()
plt.show()

## 5. Optimal allocation

In [ ]:
alloc = pd.Series(max_sharpe['weights'], index=asset_cols).sort_values(ascending=False)
alloc_pct = (alloc * 100).round(2)
alloc_pct

## 6. Risk metrics: VaR & CVaR + backtest vs benchmark

In [ ]:
opt_summary = backtest_summary(asset_returns, max_sharpe['weights'], 'Optimized Portfolio')
bench_summary = backtest_summary(bench_returns, np.array([1.0]), BENCHMARK_NAME)

summary_df = pd.DataFrame({
    'Optimized Portfolio': {k: v for k, v in opt_summary.items() if k not in ('label', 'growth_curve')},
    BENCHMARK_NAME: {k: v for k, v in bench_summary.items() if k not in ('label', 'growth_curve')},
})
summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
dates = prices.index[1:]
ax.plot(dates, (opt_summary['growth_curve']-1)*100, label='Optimized Portfolio', linewidth=2)
ax.plot(dates, (bench_summary['growth_curve']-1)*100, label=BENCHMARK_NAME, linewidth=2, linestyle='--')
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_ylabel('Cumulative Return (%)'); ax.set_title('Backtest: Optimized Portfolio vs Benchmark')
ax.legend()
plt.show()

## 7. Conclusion

The Monte-Carlo-optimized, max-Sharpe portfolio outperformed the benchmark on total return over the backtest window, at the cost of higher volatility and tail risk (VaR/CVaR) — see the top-level `README.md` for the full discussion and the pre-rendered result figures in `results/`.